# Severity Ranking Framework

This notebook is a starter workflow for the complaint severity section of the project.

The goal is to rank vehicle complaints by how likely they are to include a high-risk safety signal. We are not trying to prove that a vehicle has a confirmed defect. We are using the complaint data to find cases that deserve closer review.

Main target:
- `severity_primary_flag`: `True` when a complaint includes deaths, injuries, fire, or crash signals

Optional sensitivity target:
- `severity_broad_flag`: `True` when a complaint includes the primary signals plus severity-adjacent signals such as medical attention or towing required

Why this is a ranking problem:
- severe complaints are uncommon, so plain accuracy can look good even for a weak model
- we care whether the model puts truly severe complaints near the top of a review list

Output written by this notebook:
- `data/outputs/severity_partner_results.csv`

## 1. Setup

Run this cell first. It imports the packages used in the notebook and finds the repo folders automatically. Make sure your venv is setup and choose that as the kernel when prompted.

If this cell fails, make sure you opened the notebook from the repo and completed the README setup steps.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Make sure correct project root is found so all relative paths work
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
# These are the local modules I made for this project, they are not standard libraries or installed packages
from src.config import settings
from src.config.paths import OUTPUTS_DIR, PROCESSED_DATA_DIR

RANDOM_SEED = settings.RANDOM_SEED
sns.set_theme(style='whitegrid')
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Processed data folder:', PROCESSED_DATA_DIR)
print('Outputs folder:', OUTPUTS_DIR)

## 2. Load The Severity Table

This notebook starts from the cleaned case-level severity table. That means the raw complaint files and cleaning rules have already been handled by the src pipeline.

In [ ]:
# Verify the processed data is available and load it
severity_path = PROCESSED_DATA_DIR / 'odi_severity_cases.parquet'
if not severity_path.exists():
    raise FileNotFoundError(
        f'Missing {severity_path}. Run the README pipeline setup first so data/processed contains the severity table.'
    )

# Load the severity data and convert ldate into a real datatime
df = pd.read_parquet(severity_path)
df['ldate'] = pd.to_datetime(df['ldate'])

# Create a cleaned text column for the complaint description, null values are replaced with empty strings, multiple spaces are reduced to single spaces, and leading/trailing whitespace is removed
df['cdescr_model_text'] = df['cdescr'].fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

print('Rows:', len(df))
print('Columns:', len(df.columns))
display(df[['odino', 'ldate', 'mfr_name', 'maketxt', 'modeltxt', 'yeartxt', 'severity_primary_flag', 'severity_broad_flag']].head())

## 3. Understand The Target

`severity_primary_flag` is uncommon. That is why this notebook reports ranking metrics such as PR-AUC (like AUC score from an ROC Curve, but uses Precision and Recall on the axes instead of True Positive Rate and False Positive Rate) and recall in the highest-risk rows, not just accuracy.

In [ ]:
# TODO: Choose the target you want to model first. Use 'severity_primary_flag' for the main target or 'severity_broad_flag' for the broader sensitivity target.
TARGET_COL = None

if TARGET_COL is None:
    raise ValueError("TODO: Set TARGET_COL to 'severity_primary_flag' or 'severity_broad_flag' before continuing.")

if TARGET_COL not in {'severity_primary_flag', 'severity_broad_flag'}:
    raise ValueError("TARGET_COL must be 'severity_primary_flag' or 'severity_broad_flag'.")

target_summary = pd.DataFrame({
    'target': ['severity_primary_flag', 'severity_broad_flag'],
    'positive_cases': [int(df['severity_primary_flag'].sum()), int(df['severity_broad_flag'].sum())],
    'positive_rate': [float(df['severity_primary_flag'].mean()), float(df['severity_broad_flag'].mean())]
})
display(target_summary)

year_summary = df.assign(year=df['ldate'].dt.year).groupby('year').agg(
    rows=('odino', 'count'),
    primary_rate=('severity_primary_flag', 'mean'),
    broad_rate=('severity_broad_flag', 'mean')
).reset_index()
display(year_summary)

## 4. Create Time-Aware Splits

The model should learn from older complaints and be tested on later complaints. Do not randomly shuffle rows across years for the final evaluation because that would make the task easier than real future use.

For this notebook:
- train: complaints received through the end of 2024
- validation: complaints received in 2025
- holdout: complaints received in 2026, kept separate until the final comparison

In [ ]:
# TODO: - Separate the data into train, validation, and holdout sets based on the ldate column. Train set will ldate up to and including 2024-12-31,
#   val set will be ldate in 2025, and holdout will be ldate in 2026 and beyond. After splitting, print out the number ofrows and positive rates for
#   each set to verify the splits look reasonable. Use pd.Timestamp when creating the date thresholds to make it easier.

train_mask = None
valid_mask = None
holdout_mask = None

train_df = None
valid_df = None
holdout_df = None

split_summary = pd.DataFrame({
    'split': ['train_through_2024', 'valid_2025', 'holdout_2026'],
    'rows': [len(train_df), len(valid_df), len(holdout_df)],
    'positive_rate': [train_df[TARGET_COL].mean(), valid_df[TARGET_COL].mean(), holdout_df[TARGET_COL].mean()]
})
display(split_summary)

## 5. Metric Helpers

PR-AUC is the main metric here. It asks whether severe cases get higher scores than non-severe cases, which is more useful than accuracy when severe cases are rare.

`recall_top_5pct` answers a practical question: if we reviewed only the top 5% highest-risk complaints, what share of all severe complaints would we catch?
The same logic applies to the different percent scores.

Brier score is a proper scoring function that measures how accurately binary probabilistic events are predicted. A score closer to 0 is better and it's generally used to compare how well calibrated different models are (as opposed to something like F1 score which is better at telling you how well you decisions within a model perform). We'll use it to compare benchmark models against a baseline model.

In [ ]:
severity_results = []

# Use this function to get the positive class scores from a model, it will use predict_proba if available and
# fall back to decision_function with a sigmoid transformation if not
def get_positive_scores(model, X):
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X)
        return proba[:, 1]
    scores = model.decision_function(X)
    return 1 / (1 + np.exp(-scores))

# This function calculates the recall at the top fraction of cases based on the model's scores. It sorts the scores,
# selects the top fraction, and computes how many of the true positives are captured in that top fraction compared to
# the total number of positives.
def recall_at_top_fraction(y_true, score, fraction=0.05):
    y_true = np.asarray(y_true).astype(bool)
    score = np.asarray(score)
    n_top = max(1, int(np.ceil(len(score) * fraction)))
    top_idx = np.argsort(score)[-n_top:]
    total_positive = y_true.sum()
    if total_positive == 0:
        return np.nan
    return y_true[top_idx].sum() / total_positive

# This function evaluates the full set of metric for a give model and dataset then appends the results to the severity_results list
def evaluate_scores(model_name, split_name, y_true, score, threshold=0.5):
    pred = score >= threshold
    row = {
        'model': model_name,
        'split': split_name,
        'rows': len(y_true),
        'positive_rate': float(np.mean(y_true)),
        'pr_auc': float(average_precision_score(y_true, score)),
        'roc_auc': float(roc_auc_score(y_true, score)),
        'f1_at_0_5': float(f1_score(y_true, pred, zero_division=0)),
        'precision_at_0_5': float(precision_score(y_true, pred, zero_division=0)),
        'recall_at_0_5': float(recall_score(y_true, pred, zero_division=0)),
        'recall_top_5pct': float(recall_at_top_fraction(y_true, score, 0.05)),
        'recall_top_10pct': float(recall_at_top_fraction(y_true, score, 0.10)),
        'brier_score': float(brier_score_loss(y_true, score))
    }
    severity_results.append(row)
    return row

def show_results():
    results_df = pd.DataFrame(severity_results)
    display(results_df.sort_values(['split', 'pr_auc'], ascending=[True, False]))

## 6. Naive Baseline

A baseline is a simple model we must beat. If our real models cannot beat it, they are not useful.

This one uses a simple dummy model that predicts the same probability for all cases based on the overall positive rate in the training data. We will use the DummyClassifier from scikit-learn with the 'prior' strategy, which means it will predict the class probabilities based on the distribution of classes in the training data (i.e. for our binary example, if 1% of the target variable is True then DummyClassifier will use predict_proba to match that distribution). After fitting the dummy model, we will get the predicted probabilities for the validation set and evaluate the metrics using our evaluate_scores function.

In [ ]:
dummy_model = DummyClassifier(strategy='prior', random_state=RANDOM_SEED)
dummy_model.fit(train_df[['odino']], train_df[TARGET_COL])

dummy_valid_score = dummy_model.predict_proba(valid_df[['odino']])[:, 1]
dummy_scores = evaluate_scores('dummy_prior', 'valid_2025', valid_df[TARGET_COL].to_numpy(), dummy_valid_score)
show_results()

## 7. Structured Baseline

This model uses vehicle and complaint metadata such as manufacturer, make, model, state, mileage, speed, and timing fields.

It uses `class_weight='balanced'` because severe complaints are uncommon. That tells the model to pay more attention to the rare positive cases.

In [ ]:
# All the categorical features in the dataset we're interested in
cat_features = [
    'mfr_name',
    'maketxt',
    'modeltxt',
    'state',
    'cmpl_type',
    'drive_train',
    'fuel_sys',
    'fuel_type',
    'trans_type',
    'fire',
    'crash',
    'medical_attn',
    'vehicles_towed_yn',
    'police_rpt_yn',
    'repaired_yn',
    'source_era',
]

# All the numeric features in the dataset we're interested in
num_features = [
    'yeartxt',
    'miles',
    'veh_speed',
    'injured',
    'deaths',
    'lag_days_safe',
    'miles_zero_flag',
    'veh_speed_zero_flag',
    'miles_missing_flag',
    'veh_speed_missing_flag',
    'faildate_trusted_flag',
]

# Checks that the features above are actually in the dataframe and only keeps those that are
cat_features = [c for c in cat_features if c in df.columns]
num_features = [c for c in num_features if c in df.columns]

# Converts pandas nullable values into plain strings/floats so sklearn does not trip over pd.NA
def prepare_structured_features(source_df):
    X = source_df[cat_features + num_features].copy()
    for col in cat_features:
        X[col] = X[col].astype('string').fillna('missing').astype(str)
    for col in num_features:
        X[col] = pd.to_numeric(X[col], errors='coerce').astype('float64')
    return X

X_train_structured = prepare_structured_features(train_df)
X_valid_structured = prepare_structured_features(valid_df)
X_holdout_structured = prepare_structured_features(holdout_df)

# Creates a preprocessing pipeline for the structured features, with transformations for both categorical and numeric features
structured_preprocess = ColumnTransformer(
    transformers=[
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=50))
        ]), cat_features),
        ('num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler(with_mean=False))
        ]), num_features)
    ],
    remainder='drop'
)

# Creates a modeling pipeline that first uses the preprocessing pipeline then uses a linear Stochastic Gradient Descent model
# This works similarly to Logistic Regression but is optimized for large datasets like ours, especially with sparse features
structured_model = Pipeline([
    ('prep', structured_preprocess),
    ('model', SGDClassifier(
        loss='log_loss',
        penalty='l2',
        alpha=1e-5,
        max_iter=1000,
        tol=1e-3,
        class_weight='balanced',
        random_state=RANDOM_SEED
    ))
])

structured_model.fit(X_train_structured, train_df[TARGET_COL])
structured_valid_score = get_positive_scores(structured_model, X_valid_structured)
evaluate_scores('structured_sgd', 'valid_2025', valid_df[TARGET_COL].to_numpy(), structured_valid_score)
show_results()

## 8. Text Baseline

This model uses the complaint narrative text. TF-IDF turns the text into numbers by giving more weight to words and phrases that are distinctive.

This section now uses two text views:
- word TF-IDF catches words and short phrases such as `air bag`, `caught fire`, or `brake failure`
- character TF-IDF catches smaller spelling patterns and helps when narratives contain typos or inconsistent wording

This is often a strong baseline because people describe crashes, fires, injuries, and urgent failures directly in the narrative.

In [ ]:
# TODO: Choose text settings. Good starter values are WORD_NGRAM_RANGE = (1, 2), CHAR_NGRAM_RANGE = (3, 5), and TEXT_MIN_DF = 5.
WORD_NGRAM_RANGE = None
CHAR_NGRAM_RANGE = None
TEXT_MIN_DF = None

if WORD_NGRAM_RANGE is None or CHAR_NGRAM_RANGE is None or TEXT_MIN_DF is None:
    raise ValueError('TODO: Fill in WORD_NGRAM_RANGE, CHAR_NGRAM_RANGE, and TEXT_MIN_DF before training the text model.')

train_text = train_df['cdescr_model_text']
valid_text = valid_df['cdescr_model_text']
holdout_text = holdout_df['cdescr_model_text']

# FeatureUnion combines two text preprocessing pipelines into one feature table
text_preprocess = FeatureUnion([
    ('word_tfidf', TfidfVectorizer(
        analyzer='word',
        ngram_range=WORD_NGRAM_RANGE,
        min_df=TEXT_MIN_DF,
        max_df=0.995,
        sublinear_tf=True,
        strip_accents='unicode',
        lowercase=True,
        dtype=np.float32
    )),
    ('char_tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=CHAR_NGRAM_RANGE,
        min_df=TEXT_MIN_DF,
        sublinear_tf=True,
        strip_accents='unicode',
        lowercase=True,
        dtype=np.float32
    ))
])

text_model = Pipeline([
    ('tfidf', text_preprocess),
    ('model', SGDClassifier(
        loss='log_loss',
        penalty='l2',
        alpha=1e-5,
        max_iter=1000,
        tol=1e-3,
        class_weight='balanced',
        random_state=RANDOM_SEED
    ))
])

text_model.fit(train_text, train_df[TARGET_COL])
text_valid_score = get_positive_scores(text_model, valid_text)
evaluate_scores('text_tfidf_word_char_sgd', 'valid_2025', valid_df[TARGET_COL].to_numpy(), text_valid_score)
show_results()

## 9. Optional: Final Holdout Check

Only run this after choosing your best model from the 2025 validation results. The holdout is meant to be a final check on 2026 complaints.

In [ ]:
# TODO: After evaluating models above, change RUN_HOLDOUT to TRUE and BEST_MODEL_NAME to the name of the model with the best results metrics (match names in model_lookup below)
RUN_HOLDOUT = False
BEST_MODEL_NAME = None

if RUN_HOLDOUT:
    model_lookup = {
        'dummy_prior': dummy_model,
        'structured_sgd': structured_model,
        'text_tfidf_word_char_sgd': text_model
    }
    input_lookup = {
        'dummy_prior': holdout_df[['odino']],
        'structured_sgd': X_holdout_structured,
        'text_tfidf_word_char_sgd': holdout_text
    }
    best_model = model_lookup[BEST_MODEL_NAME]
    holdout_score = get_positive_scores(best_model, input_lookup[BEST_MODEL_NAME])
    evaluate_scores(BEST_MODEL_NAME, 'holdout_2026', holdout_df[TARGET_COL].to_numpy(), holdout_score)
    show_results()
else:
    print('Holdout skipped. Set RUN_HOLDOUT = True only after choosing the best model from validation results.')

## 10. Save Results

This cell writes a small CSV summary that can be shared with the team.

In [ ]:
results_df = show_results()
results_path = OUTPUTS_DIR / 'severity_partner_results.csv'
results_df.to_csv(results_path, index=False)
print('Wrote:', results_path)

## 11. Next TODOs / Extension Ideas

Use this section as a checklist for improving the severity ranking notebook after the starter baselines run.

Suggested next steps:
- Try `severity_broad_flag` as the target and compare it with `severity_primary_flag`. The broad target will usually catch more cases but may be less specific.
- Add a combined text + structured model with `ColumnTransformer` or `FeatureUnion`. The goal is to see whether complaint text plus vehicle/context fields beats either source by itself.
- Add a simple threshold-tuning cell. For example, pick the score cutoff that gives a useful precision/recall tradeoff on `valid_2025`.
- Add a top-risk review table that shows the highest-scored validation complaints with `odino`, `ldate`, severity flags, model score, and `cdescr_model_text`.
- Add a calibration plot or Brier score check if you want to talk about whether the risk scores behave like probabilities.
- Add one or two charts, such as severity rate by year or score distribution by target value.
- If the model looks useful, run the optional holdout cell once with the best model name and save the final comparison.

Keep the holdout set for the final check only. Use `valid_2025` for experimenting with features, thresholds, and model settings.